# 蜃景 × lightx2v —— 纯 t2v 文生视频部署（无 ComfyUI）

为「纯 t2v + lightx2v、彻底不用 ComfyUI」准备。lightx2v 是你那套 4 步蒸馏 LoRA 的娘家、专做 Wan 快速推理。
**本笔记本不装 ComfyUI、不出图、不锁脸、不换脸**——只做：文本→视频(lightx2v) + 配音 + 字幕。

**⚠️ 关键未验证项（先跑 §3/§4/§5 确认，再往后）：** lightx2v 在 Blackwell(sm_120) 上能否装上
（尤其注意力算子）、server 能否起来。装不上时优先走它的 **nvfp4** 路线（不依赖 SageAttention）。
带 ★ 的命令是按 lightx2v 文档写的**脚手架**，以你的 lightx2v README/scripts 实际为准、按需改。

**用法**：运行时选 GPU → 逐格 Run。角色 LoRA 训练仍在 `colab_deploy.ipynb` 的 LW1/LW2（用 ai-toolkit，与本部署独立）。

In [ ]:
# === §1 参数 + Drive + GPU 探测（设 T2V_PRECISION）===
REPO_URL = 'https://github.com/aw3n1998/build-with-langchain.git'
BRANCH   = 'main'
import os, torch as _t
os.makedirs('/content', exist_ok=True)
if _t.cuda.is_available():
    cc = _t.cuda.get_device_capability(0); name = _t.cuda.get_device_name(0)
    vram = _t.cuda.get_device_properties(0).total_memory / (1024**3)
    native_fp8 = (cc[0] > 8) or (cc[0] == 8 and cc[1] >= 9)
    # 原生 fp8(Blackwell/Hopper/Ada)→ lightx2v 走 fp8/nvfp4 最快;A100(sm80)无原生 fp8 → int8/bf16
    _prec = 'fp8' if native_fp8 else 'fp16'
    os.environ['T2V_PRECISION'] = _prec
    print(f'🖥 GPU: {name} | sm{cc[0]}.{cc[1]} | {vram:.0f}G | 原生FP8={native_fp8} → T2V_PRECISION={_prec}')
    if cc[0] >= 12:
        print('   ★Blackwell(sm120):lightx2v 注意力算子可能要预编 wheel 或走 nvfp4;§3 留意报错。')
else:
    os.environ['T2V_PRECISION'] = 'fp16'
    print('⚠️ 没检测到 GPU！代码执行程序 → 更改运行时类型 → 选 GPU')
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'; CACHE=DRIVE+'/mirage_models'; TOOLS=DRIVE+'/mirage_tools'
PIP_CACHE=TOOLS+'/pip_cache'
for p in (CACHE, TOOLS, PIP_CACHE): os.makedirs(p, exist_ok=True)
os.environ['PIP_CACHE_DIR']=PIP_CACHE; os.environ['HF_HOME']=TOOLS+'/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('环境就绪 | 模型:', CACHE)

In [ ]:
# === §2 拉取/更新蜃景仓库 ===
import os, sys
%cd /content
if not os.path.isdir('mirage/.git'):
    !git clone -b {BRANCH} {REPO_URL} mirage
else:
    !cd mirage && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
%cd /content/mirage
sys.path.insert(0, '/content/mirage/colab')
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in sys.path if p)  # 子进程继承内核 torch(sm120 必需)
print('仓库就绪 @', BRANCH)

In [ ]:
# === §3 装 lightx2v（★sm_120 上的关键关：注意力算子装不上就走 nvfp4）===
# 核心框架:pip 装。注意力算子(SageAttention/FlashAttention)按 lightx2v 文档单独装——这步是 Blackwell 风险点。
!pip -q install -v "git+https://github.com/ModelTC/lightx2v.git" || echo '⚠️ lightx2v 主框架装失败,看上面报错'
# ★注意力算子(以 lightx2v 文档为准;Blackwell 优先找预编 wheel 或 nvfp4 路线):
# !pip -q install sageattention   # 或 flash-attn,sm120 多半要对应 torch 的预编 wheel
import importlib.util as _u
print('lightx2v 可导入:' , bool(_u.find_spec('lightx2v')))
print('sageattention 可用:', bool(_u.find_spec('sageattention')), '(没有就走 nvfp4,见 lightx2v 文档)')

In [ ]:
# === §4 下 T2V 权重：官方 base(Wan2.2-T2V-A14B) + lightx2v 4步蒸馏 LoRA（★过滤下载，绝不下整仓★）===
# ⚠️ lightx2v 对 T2V 没出「独立蒸馏整模型」(那是 I2V 才有)，T2V 只发了 4步蒸馏 LoRA(在 Wan2.2-Lightning 仓)。
#    所以 T2V = 官方 base(Wan-AI/Wan2.2-T2V-A14B 高/低噪双专家) + 这个小 LoRA(high/low 各一个)。
#    base 仓 usedStorage 240G(含历史)，但当前需要的那套(双专家 bf16 + VAE + 文本编码器)≈ 67G —— 用 --include 过滤，别下整仓。
import os
CACHE = '/content/drive/MyDrive/mirage_models'
BASE_T2V = CACHE + '/wan2.2_t2v_a14b'          # 官方 base（lightx2v 要的目录格式：high_noise_model/ + low_noise_model/）
LORA_DIR = CACHE + '/lightx2v_t2v_lora'        # 4步蒸馏 LoRA(小)
os.makedirs(BASE_T2V, exist_ok=True); os.makedirs(LORA_DIR, exist_ok=True)

# 1) 4步蒸馏 LoRA(小，先下)：lightx2v/Wan2.2-Lightning 最新 T2V rank64
_LORA_SUB = 'Wan2.2-T2V-A14B-4steps-lora-rank64-Seko-V2.0'
!hf download lightx2v/Wan2.2-Lightning --include "{_LORA_SUB}/*" --local-dir "{LORA_DIR}" || echo '⚠️ LoRA 没下到，核对 huggingface.co/lightx2v/Wan2.2-Lightning 子目录名'
os.environ['LIGHTX2V_DISTILL_LORA_HIGH'] = f'{LORA_DIR}/{_LORA_SUB}/high_noise_model.safetensors'
os.environ['LIGHTX2V_DISTILL_LORA_LOW']  = f'{LORA_DIR}/{_LORA_SUB}/low_noise_model.safetensors'

# 2) 官方 base(~67G；★只取双专家+VAE+bf16 文本编码器，不下整仓 240G 历史★)
!hf download Wan-AI/Wan2.2-T2V-A14B --include "high_noise_model/*" "low_noise_model/*" "*.json" "Wan2.1_VAE.pth" "models_t5_umt5-xxl-enc-bf16.pth" --local-dir "{BASE_T2V}" || echo '⚠️ base 没下到，核对仓库名'
os.environ['LIGHTX2V_MODEL_T2V'] = BASE_T2V
print('T2V base:', BASE_T2V, '| 蒸馏 LoRA:', _LORA_SUB, '| 精度档:', os.environ.get('T2V_PRECISION'))

In [ ]:
# === §5 起 lightx2v HTTP server（★启动命令以 lightx2v scripts/server 为准；端口 8189）===
# lightx2v 自带 FastAPI server。clone 其仓库拿 scripts/server,后台起服务(脱离内核,停 cell 不杀)。
import os, sys; sys.path.insert(0, '/content/mirage/colab'); import persist
if not os.path.isdir('/content/LightX2V/.git'):
    !git clone https://github.com/ModelTC/lightx2v /content/LightX2V
# ★真实启动方式以 /content/LightX2V/scripts/server/ 与其 README 为准。下面是脚手架占位:
#   通常需要一个 config(指定 model_cls=wan2.2 / task=t2v / model_path=$LIGHTX2V_MODEL_T2V / port=8189)。
os.environ['LIGHTX2V_PORT'] = '8189'
print('★请按 lightx2v README 在此起 server,例如:')
print('   cd /content/LightX2V && bash scripts/server/start_server.sh   # 配 model_path/port/task=t2v')
print('   起好后健康检查:  curl http://127.0.0.1:8189/v1/service/status')
# 起好后填:
os.environ['LIGHTX2V_BASE_URL'] = 'http://127.0.0.1:8189'
# persist.start_bg([...lightx2v server 启动命令...], '/content/lightx2v.log')   # ← 核对命令后启用
print('LIGHTX2V_BASE_URL =', os.environ['LIGHTX2V_BASE_URL'])

In [ ]:
# === §6 装蜃景后端依赖 + 写 .env（不配 ComfyUI → 出图/换脸不注册；t2v 走 lightx2v）===
%cd /content/mirage
!pip -q install -r requirements.txt
!pip -q install fastembed edge-tts
import os
lines = [
    'COMFYUI_BASE_URL=',                                  # ★空 = 不起/不连 ComfyUI(纯 t2v)
    'LIGHTX2V_ENABLED=true',
    'LIGHTX2V_BASE_URL=' + os.environ.get('LIGHTX2V_BASE_URL', 'http://127.0.0.1:8189'),
    'LIGHTX2V_MODEL_T2V=' + os.environ.get('LIGHTX2V_MODEL_T2V', ''),          # §4 下的官方 base 目录
    'LIGHTX2V_DISTILL_LORA_HIGH=' + os.environ.get('LIGHTX2V_DISTILL_LORA_HIGH', ''),  # §4 下的 4步蒸馏 LoRA(高噪)
    'LIGHTX2V_DISTILL_LORA_LOW=' + os.environ.get('LIGHTX2V_DISTILL_LORA_LOW', ''),    # §4 下的 4步蒸馏 LoRA(低噪)
    'T2V_PROVIDER=lightx2v-t2v',                          # t2v 出片走 lightx2v
    'T2V_PRECISION=' + os.environ.get('T2V_PRECISION', 'fp8'),
    'VIDEO_PROVIDER_DEFAULT=wan2.2',
]
open('.env', 'w').write('\n'.join(lines) + '\n')
print('.env 写好(纯 t2v + lightx2v,无 ComfyUI):')
print('\n'.join('  ' + l for l in lines))

In [ ]:
# === §7 起蜃景后端（读 §6 的 .env）===
import sys; sys.path.insert(0, '/content/mirage/colab'); import persist
persist.restart_service('后端',
    ['uvicorn','mirage.main_api:app','--host','0.0.0.0','--port','8000'],
    'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn',
    cwd='/content/mirage', timeout=120)

In [ ]:
# === §8 cloudflared 暴露后端（拿公网地址）===
import sys, re, time, os, subprocess
sys.path.insert(0, '/content/mirage/colab'); import persist
subprocess.run('pkill -9 -f cloudflared 2>/dev/null; sleep 2', shell=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
                   'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True)
open('/content/cf.log', 'w').close()
persist.start_bg(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], '/content/cf.log')
url = None
for _ in range(60):
    time.sleep(1)
    m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m: url = m[-1]; break
print('✅ 公网地址:', url) if url else print('⚠ 60s 未就绪 → !tail -20 /content/cf.log')
print('打开它 → 工作台「出片模式 = t2v」→ 一键全自动出片(走 lightx2v,无 ComfyUI)。配音+字幕自动。')

In [ ]:
# === §9 实时日志（看 lightx2v 出片采样到第几步；tail -f 阻塞，停本格只停 tail、不杀服务）===
# 看到采样进度 % = 正在出片；卡住 / 报错也在这显形。
# 出片走 lightx2v server → 日志在 /content/lightx2v.log（★确保 §5 起 server 时把日志写到这个文件）。
# 后端日志看 /content/api.log；隧道日志看 /content/cf.log。要看哪个把下面文件名换掉即可。
import os
_log = '/content/lightx2v.log'
open(_log, 'a').close()   # 没有就建个空文件，免 tail 报错
!tail -n 80 -f {_log}


In [ ]:
# === §10. (可选) 启用「网页里点开始训练」= 装 ai-toolkit + 下 Wan-T2V 训练底模 + 配 .env ===
# 想在工作台「角色 & LoRA」里直接训角色 LoRA(Wan-T2V)才跑这格。★训练底模 diffusers ~56G、与 §4 出片底模是两套★。
# 盘紧/嫌重 → 改用 colab_deploy.ipynb 的 LW1/LW2 训，训完把 loras/*_wan_t2v_high/low 拷过来 + 在项目里填上即可。
import os, re, subprocess, sys
sys.path.insert(0, '/content/mirage/colab'); import persist
def _sh(c): r = subprocess.run(c, shell=True, capture_output=True, text=True); return r.stdout + r.stderr
TOOLS = '/content/drive/MyDrive/mirage_tools'; AITK_DRIVE = TOOLS + '/ai-toolkit'; AITK = '/content/ai-toolkit'
WAN_BASE = TOOLS + '/wan22_t2v_diffusers'                 # ai-toolkit 训练底模(diffusers,~56G),持久化 Drive
# 1) ai-toolkit(持久化,不重 clone)
if not os.path.isdir(AITK_DRIVE + '/.git'):
    print(_sh(f'git clone https://github.com/ostris/ai-toolkit {AITK_DRIVE} && cd {AITK_DRIVE} && git submodule update --init --recursive')[-300:])
else:
    print(_sh(f'cd {AITK_DRIVE} && git pull -q && git submodule update --init --recursive -q'))
persist.link_dir(AITK_DRIVE, AITK)
print(_sh(f'cd {AITK} && pip -q install -r requirements.txt')[-300:])
# 2) Wan-T2V 训练底模(★只在缺时下,~56G★)
os.makedirs(WAN_BASE, exist_ok=True)
if not os.path.exists(WAN_BASE + '/model_index.json'):
    print('下 Wan2.2-T2V-A14B diffusers 训练底模(~56G,慢)…'); print(_sh(f'hf download ai-toolkit/Wan2.2-T2V-A14B-Diffusers-bf16 --local-dir {WAN_BASE}')[-300:])
else:
    print('训练底模已在:', WAN_BASE)
# 3) 写 .env：训练执行器指本地 ai-toolkit + 本地 Wan 底模(免训练时重下 56G)
envp = '/content/mirage/.env'; env = open(envp).read() if os.path.exists(envp) else ''
def _se(e, k, v): return re.sub(k + r'=.*', k + '=' + v, e) if (k + '=') in e else e + ('' if e.endswith(chr(10)) else chr(10)) + k + '=' + v + chr(10)
for k, v in [('AI_TOOLKIT_DIR', AITK), ('LORA_TRAIN_BASE', WAN_BASE), ('COMFYUI_LORA_DIR', '/content/ComfyUI/models/loras')]:
    env = _se(env, k, v)
open(envp, 'w').write(env)
persist.restart_service('后端', ['uvicorn', 'mirage.main_api:app', '--host', '0.0.0.0', '--port', '8000'],
    'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn', cwd='/content/mirage', timeout=120)
print('✅ 网页「角色 & LoRA」→ 传 16-20 张同脸图 → 开始训练 = 训 Wan-T2V LoRA(high+low)，训完自动挂本剧 t2v 出片。')
print('⚠️ 造图模式(免上传自训)要 ComfyUI，纯 t2v 没有 → 手动上传训练图。')

---
## 说明
- **本部署纯 t2v**：没有出图/选图/PuLID/换脸（那些要 ComfyUI）。身份靠训好的 Wan-T2V 角色 LoRA。
- **角色 LoRA 训练**：仍用 `colab_deploy.ipynb` 的 **LW1/LW2**（ai-toolkit，与 lightx2v 推理独立）；
  训好的 LoRA 放进 lightx2v 的 lora_configs（后端 `WAN_T2V_LORA_HIGH/LOW` 或项目设置）。
- **配音 + 字幕**：合成时自动（edge-tts 按角色音色 + 烧字幕），与后端无关。
- **★首跑核对清单**：§3 lightx2v + 注意力算子在 Blackwell 装上了吗 → §4 权重仓库名对吗 →
  §5 server 起命令对吗、`/v1/service/status` 通吗 → 出一条片确认 `providers/lightx2v.py` 的请求/取片字段对得上
  （对不上就按真机改 `_extract_output` / payload）。